`neighborhood_indicators.csv` contains an enormous volume of data we don't actually care about. This notebook serves to parse out the specific bit we do want, namely the `crime_all_rt` column. It also filters out the rows sorted by geographical units other than community districts, and creates five-year blocks to match the format of most of our other datasets.

Note that this code assumes that the CSV files are located *outside* the folder containing this notebook. This is because my local setup consists of all the CSV files next to the top-level folder for the git repo, so that I don't have to worry about putting them into the repo folder and then `.gitignore`ing them.

In [11]:
import pyspark.sql.functions as sf
from pyspark.sql import SparkSession
from pyspark.sql.window import Window

In [10]:
spark = SparkSession.builder.appName("crime_parse").getOrCreate()

indicDF = spark.read.format("csv") \
            .option("header", True) \
            .option("infer_schema", True) \
            .load("../neighborhood_indicators.csv")

In [38]:
trimmed = indicDF.filter("region_type = 'Community District'") \
                 .filter("crime_all_rt IS NOT NULL") \
                 .select("region_id", "year", "crime_all_rt")

lagWindow = Window.partitionBy("region_id").orderBy("year")
blockAvgWindow = lagWindow.rowsBetween(-4, 0)
blockified = trimmed.select(
    "region_id",
    sf.concat(
        sf.to_char(sf.lag("year", 4).over(lagWindow), sf.lit("9999")),
        sf.lit("-"),
        sf.to_char(sf.try_subtract("year", sf.lit(2000)), sf.lit("09"))
    ).alias("year_block"),
    sf.round(sf.avg("crime_all_rt").over(blockAvgWindow), 2).alias("avg_crime_rt")
).filter("year_block IS NOT NULL")

blockified.write.format("csv").mode("overwrite").save("../new_tables/crime_rate", header=True)